In [1]:
import pandas as pd


In [2]:
data = pd.read_csv('creditcard_pra.csv')

In [3]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
data.Class.value_counts()

0    284315
1       492
Name: Class, dtype: int64

In [5]:
# totally imbalanced data set.

In [6]:
X = data.drop('Class', axis = 1)
y = data.Class

* Before moving on under and over sampling we use some algo and check how its performing.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import KFold
import numpy as np
from sklearn.model_selection import GridSearchCV

In [8]:
log = LogisticRegression()
grid = {'C':10.0 **np.arange(-2,3), 'penalty': ['l1','l2']}
cv = KFold(n_splits = 5, random_state = None, shuffle = False)


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [11]:
cl = GridSearchCV(log, grid, cv = cv, n_jobs = -1, scoring = 'f1_macro' )

In [12]:
cl.fit(X_train, y_train)

C:\Users\aravi\anaconda3\envs\tensflow\lib\site-packages\sklearn\model_selection\_search.py:925: UserWarning: One or more of the test scores are non-finite: [       nan 0.82931309        nan 0.84467917        nan 0.84741908
        nan 0.83997595        nan 0.83848539]
  category=UserWarning
C:\Users\aravi\anaconda3\envs\tensflow\lib\site-packages\sklearn\linear_model\_logistic.py:765: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG)


GridSearchCV(cv=KFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'penalty': ['l1', 'l2']},
             scoring='f1_macro')

In [13]:
pred = cl.predict(X_test)

In [14]:
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

[[93799    39]
 [   61    88]]
0.9989360230670199
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.69      0.59      0.64       149

    accuracy                           1.00     93987
   macro avg       0.85      0.80      0.82     93987
weighted avg       1.00      1.00      1.00     93987



In [15]:
from sklearn.ensemble import RandomForestClassifier

In [16]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

RandomForestClassifier()

In [17]:
pred= rf.predict(X_test)
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

[[93830     8]
 [   31   118]]
0.9995850489961378
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.94      0.79      0.86       149

    accuracy                           1.00     93987
   macro avg       0.97      0.90      0.93     93987
weighted avg       1.00      1.00      1.00     93987




* Quite Fine, you can use this technique last, but before we use some feature engineering.

### Under Sampling
* Reduce the points of the maximum labels.
* If the data is small, defenitely go with under sampling

In [18]:
from imblearn.under_sampling import NearMiss

In [19]:
from collections import Counter
ns = NearMiss(0.8)
X_train_s, y_train_s = ns.fit_resample(X_train, y_train)
print('the number of classes before fit {}'.format(Counter(y_train)))
print('the number of classes before fit {}'.format(Counter(y_train_s)))

C:\Users\aravi\anaconda3\envs\tensflow\lib\site-packages\imblearn\utils\_validation.py:591: FutureWarning: Pass sampling_strategy=0.8 as keyword args. From version 0.9 passing these as positional arguments will result in an error
  FutureWarning,


the number of classes before fit Counter({0: 190477, 1: 343})
the number of classes before fit Counter({0: 428, 1: 343})


In [20]:
428*0.80

342.40000000000003

In [21]:
rf = RandomForestClassifier()
rf.fit(X_train_s,y_train_s)
pred= rf.predict(X_test)
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


[[68305 25533]
 [    9   140]]
0.7282390117782247
              precision    recall  f1-score   support

           0       1.00      0.73      0.84     93838
           1       0.01      0.94      0.01       149

    accuracy                           0.73     93987
   macro avg       0.50      0.83      0.43     93987
weighted avg       1.00      0.73      0.84     93987




### Over Sampling
* Increase the point of maximum labels.

In [22]:
from imblearn.over_sampling import RandomOverSampler

In [23]:
os = RandomOverSampler(0.5)
X_train_os, y_train_os = os.fit_resample(X_train, y_train)
print('the number of classes before fit {}'.format(Counter(y_train)))
print('the number of classes before fit {}'.format(Counter(y_train_os)))

C:\Users\aravi\anaconda3\envs\tensflow\lib\site-packages\imblearn\utils\_validation.py:591: FutureWarning: Pass sampling_strategy=0.5 as keyword args. From version 0.9 passing these as positional arguments will result in an error
  FutureWarning,


the number of classes before fit Counter({0: 190477, 1: 343})
the number of classes before fit Counter({0: 190477, 1: 95238})


In [24]:
rf = RandomForestClassifier()
rf.fit(X_train_os,y_train_os)
pred= rf.predict(X_test)
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


[[93833     5]
 [   28   121]]
0.9996488876121166
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.96      0.81      0.88       149

    accuracy                           1.00     93987
   macro avg       0.98      0.91      0.94     93987
weighted avg       1.00      1.00      1.00     93987



### SMOTETomek

In [25]:
from imblearn.combine import SMOTETomek
# it will create a new point until satisfy the maximum labels in the data.

In [26]:
os = SMOTETomek(0.5)
X_train_os, y_train_os = os.fit_resample(X_train, y_train)
print('the number of classes before fit {}'.format(Counter(y_train)))
print('the number of classes before fit {}'.format(Counter(y_train_os)))

C:\Users\aravi\anaconda3\envs\tensflow\lib\site-packages\imblearn\utils\_validation.py:591: FutureWarning: Pass sampling_strategy=0.5 as keyword args. From version 0.9 passing these as positional arguments will result in an error
  FutureWarning,


the number of classes before fit Counter({0: 190477, 1: 343})
the number of classes before fit Counter({0: 189748, 1: 94509})


In [27]:
rf = RandomForestClassifier()
rf.fit(X_train_os,y_train_os)
pred= rf.predict(X_test)
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


[[93812    26]
 [   21   128]]
0.9994999308414994
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93838
           1       0.83      0.86      0.84       149

    accuracy                           1.00     93987
   macro avg       0.92      0.93      0.92     93987
weighted avg       1.00      1.00      1.00     93987



### Ensemble Techniques

In [28]:
from imblearn.ensemble import EasyEnsembleClassifier

In [29]:
easy = EasyEnsembleClassifier()
easy.fit(X_train, y_train)

EasyEnsembleClassifier()

In [30]:
pred = easy.predict(X_test)

In [31]:
print(confusion_matrix(y_test, pred ))
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


[[90091  3747]
 [   10   139]]
0.9600263866279379
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     93838
           1       0.04      0.93      0.07       149

    accuracy                           0.96     93987
   macro avg       0.52      0.95      0.52     93987
weighted avg       1.00      0.96      0.98     93987



In [32]:
# it's performed well so we use trail and error basics